# Phase 1: Baselines

Goal: set the bar every later model has to clear, and build the evaluation harness (`training/recsys/evaluate.py`) that phases 2–4 reuse. By the end you should be able to answer:

1. How good is "predict the average rating" (RMSE)?
2. How much do user and movie biases improve on that?
3. What does regularization do to the bias model's top-10 list?
4. Does the model with the best RMSE also give the best recommendations?
5. Can a personalized-looking model beat "show everyone the most popular movies"?

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make training/ importable so the shared `recsys` package works from this notebook.
sys.path.insert(0, str(Path.cwd().resolve().parent))
from recsys.baselines import BiasModel, Popularity
from recsys.data import load_movies, load_ratings, time_split
from recsys.evaluate import evaluate, rmse

ratings = load_ratings()
movies = load_movies().set_index("movieId")
train, val, test = time_split(ratings)

for name, df in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s} {len(df):7,} ratings  {df['userId'].nunique()} users  {df['movieId'].nunique():,} movies")

## 1. The split

For each user, sort their ratings by time: the oldest 80% go to **train**, the next 10% to **val**, the newest 10% to **test**.

- **Why per user?** Phase 0 section 6 showed that most users rate everything in one sitting, so a single global cutoff date leaves most test users with no training data. That measures cold start, not recommendation quality.
- **Why val *and* test?** We choose hyperparameters (here, regularization strength) by looking at val. If we chose them by looking at test, the test score would be optimistic. Test is touched once, at the very end.

### Metrics

- **RMSE**: typical error of a predicted star rating. Lower is better.
- **Recall@10**: of the movies a user rated ≥ 4 in the held-out period, what fraction appear in our top 10?
- **NDCG@10**: like recall, but a hit at rank 1 counts more than a hit at rank 10.
- **Coverage@10**: what fraction of the catalog appears in *anyone's* top 10? Low means everyone gets the same list.

Movies a user already rated in train are never recommended back to them.

## 2. Global mean

Predict the training average for every rating. Any model that can't beat this has learned nothing.

In [ ]:
mu = train["rating"].mean()
print(f"global mean: {mu:.3f}")
print(f"val RMSE:    {rmse(val['rating'], np.full(len(val), mu)):.4f}")

## 3. Popularity

Score every movie by how many ratings it has in train. Everyone gets the same list (minus what they've already seen). It can't predict ratings, so no RMSE.

In [ ]:
pop = Popularity().fit(train)
print(evaluate(pop, train, val))

top = pop.item_ids[np.argsort(-pop._counts)[:10]]
movies.loc[top, ["title"]]

## 4. Bias model

`rating ≈ mu + b_user + b_movie`

- `b_user`: this user rates everything higher or lower than average (a tough critic has a negative bias).
- `b_movie`: this movie is rated higher or lower than average, after accounting for who rated it.

Training alternates closed-form updates (see `recsys/baselines.py`). The regularization `reg` acts like `reg` imaginary ratings at the global mean: `b_movie = sum(residuals) / (n_ratings + reg)`. A movie with 2 ratings gets pulled hard toward 0; one with 300 barely moves.

Compare the top 10 movies by `b_movie` without and with regularization:

In [ ]:
counts = train.groupby("movieId").size()

def top_movies(model: BiasModel, n: int = 10) -> pd.DataFrame:
    order = np.argsort(-model.b_item)[:n]
    ids = model.item_ids[order]
    return pd.DataFrame({
        "title": movies.loc[ids, "title"].to_numpy(),
        "b_movie": model.b_item[order].round(2),
        "n_ratings": counts.loc[ids].to_numpy(),
    })

print("reg = 0")
display(top_movies(BiasModel(reg_user=0, reg_item=0).fit(train)))
print("reg = 25")
display(top_movies(BiasModel(reg_user=25, reg_item=25).fit(train)))

## 5. Tuning regularization on val

Try a grid of `reg_user` × `reg_item` and record both RMSE and NDCG@10 on val.

In [ ]:
regs = [0, 1, 3, 5, 10, 25, 50, 100, 200]
rows = []
for reg_user in regs:
    for reg_item in regs:
        model = BiasModel(reg_user=reg_user, reg_item=reg_item).fit(train)
        rows.append({"reg_user": reg_user, "reg_item": reg_item, **evaluate(model, train, val)})
grid = pd.DataFrame(rows)

print("val RMSE (rows: reg_user, columns: reg_item)")
display(grid.pivot(index="reg_user", columns="reg_item", values="rmse").round(4))
print("val NDCG@10")
display(grid.pivot(index="reg_user", columns="reg_item", values="ndcg@10").round(4))

In [ ]:
best_rmse = grid.loc[grid["rmse"].idxmin()]
best_ndcg = grid.loc[grid["ndcg@10"].idxmax()]
print(f"best RMSE:    reg_user={best_rmse.reg_user:g}, reg_item={best_rmse.reg_item:g}  (rmse {best_rmse.rmse:.4f}, ndcg {best_rmse['ndcg@10']:.4f})")
print(f"best NDCG@10: reg_user={best_ndcg.reg_user:g}, reg_item={best_ndcg.reg_item:g}  (rmse {best_ndcg.rmse:.4f}, ndcg {best_ndcg['ndcg@10']:.4f})")

# Slice through the grid at the RMSE-best reg_user to see the trade-off.
line = grid[grid["reg_user"] == best_rmse.reg_user]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(line["reg_item"], line["rmse"], marker="o")
axes[0].set(xscale="symlog", xlabel="reg_item", ylabel="val RMSE", title="Rating prediction (lower is better)")
axes[1].plot(line["reg_item"], line["ndcg@10"], marker="o", color="tab:orange")
axes[1].axhline(evaluate(pop, train, val)["ndcg@10"], color="gray", linestyle="--", label="popularity")
axes[1].set(xscale="symlog", xlabel="reg_item", ylabel="val NDCG@10", title="Ranking (higher is better)")
axes[1].legend()
plt.show()

## 6. Final numbers on test

Refit on train + val with the chosen settings, then evaluate once on test. Train + val is now the "history" whose movies we don't recommend back.

We report two bias models, one tuned for RMSE and one tuned for NDCG, because they disagree.

In [ ]:
history = pd.concat([train, val], ignore_index=True)
models = {
    "popularity": Popularity(),
    "bias (RMSE-tuned)": BiasModel(reg_user=best_rmse.reg_user, reg_item=best_rmse.reg_item),
    "bias (NDCG-tuned)": BiasModel(reg_user=best_ndcg.reg_user, reg_item=best_ndcg.reg_item),
}
results = {"global mean": {"rmse": rmse(test["rating"], np.full(len(test), history["rating"].mean()))}}
for name, model in models.items():
    results[name] = evaluate(model.fit(history), history, test)

pd.DataFrame(results).T.round(4)

## Takeaways

Write your own answers to the five questions at the top. Some things to look for:

- How much of the RMSE gain comes from biases alone? Phase 3's matrix factorization adds personalization *on top of* these biases. Its RMSE improvement over this table is the real value of embeddings.
- Look at what `reg_item` does to the top 10: low values surface obscure movies with a few perfect ratings; high values drift toward the popularity list.
- RMSE rewards being accurate on *every* movie, including the ones a user would never pick. Ranking metrics only care about the top. A model can be good at one and bad at the other.
- Push `reg_item` past 200 and NDCG levels off at roughly the popularity line instead of peaking. With a huge `reg`, `b_movie ≈ sum(residuals) / reg`, which rewards movies with *many* above-average ratings: "popular and well-liked". That is the best ranking a bias model can produce.
- Small gaps between rows (say, under 0.005 NDCG) can be noise from which users happened to be in the split. Don't read much into them.
- These baselines show everyone the same list (coverage is tiny). The personalized models in phases 3–4 should clearly beat the popularity row on NDCG@10. If they don't, look for a bug first.

### Try next

- Change the relevance threshold in `evaluate` from 4.0 to 3.5 or 4.5. Do the rankings of the models change?
- Evaluate only on users with few training ratings (say, under 30). Which baseline holds up better?